# Preprocesamiento del Conjunto de Datos (Versión 2)

Este notebook implementa el pipeline de preprocesamiento dual, dividiendo el flujo en dos ramas tras una limpieza global inicial:
- **Pipeline Clásico:** Para algoritmos como SVM, Random Forest y Naive Bayes. Incluye tokenización, eliminación de stopwords, lematización, lowercasing y protección de signos de interrogación y exclamación.
- **Pipeline Transformers:** Para modelos tipo BERT. Omite lematización, no cambia a minúsculas y respeta la puntuación nativa.

Ambos inician tras un proceso global de demojización (con formato `emoji_nombre`) y limpieza de entidades de redes sociales (URLs y menciones).

## 1. Configuración de Entorno e Instalación

In [1]:
!pip install pandas spacy emoji
!python -m spacy download es_core_news_sm

     ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
     ---- ----------------------------------- 1.6/12.9 MB 10.5 MB/s eta 0:00:02
     ----------- ---------------------------- 3.7/12.9 MB 11.5 MB/s eta 0:00:01
     ------------------ --------------------- 6.0/12.9 MB 10.0 MB/s eta 0:00:01
     -------------------------- ------------- 8.4/12.9 MB 10.2 MB/s eta 0:00:01
     --------------------------------- ----- 11.0/12.9 MB 10.7 MB/s eta 0:00:01
     --------------------------------------  12.8/12.9 MB 10.7 MB/s eta 0:00:01
     --------------------------------------- 12.9/12.9 MB 10.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')


## 2. Carga de Librerías y Modelos

In [2]:
import pandas as pd
import re
import spacy
import emoji
import os

# Cargar el modelo en español de Spacy
nlp = spacy.load("es_core_news_sm")

# ---------------------------------------------------------
# CONFIGURACIÓN DE STOPWORDS PERSONALIZADAS
# ---------------------------------------------------------
# Protegemos palabras clave para la clasificación de emociones:
# negaciones, adverbios de cantidad/frecuencia y adjetivos importantes.
excepciones_stopwords = {
    "no", "ni", "nunca", "jamás", "tampoco", "nada", "nadie", "ninguno", "ninguna",
    "muy", "más", "poco", "menos", "bastante", "demasiado", "si", "sí", "siempre",
    "bien", "mal", "mejor", "peor", "bueno", "malo", "excelente", "terrible", "fatal"
}

for word in excepciones_stopwords:
    if word in nlp.Defaults.stop_words:
        # Retirar estas palabras de las stopwords predeterminadas de spacy
        nlp.vocab[word].is_stop = False

print("Librerías y modelos cargados correctamente.")

Librerías y modelos cargados correctamente.


## 3. Carga de los Datasets

In [3]:
rutas = [
    "scrapping-youtube-g4/comentarios_clasificado_parte01.csv",
    "scrapping-youtube-g4/comentarios_clasificado_parte02.csv",
    "scrapping-tiktok-g4/comentarios_clasificado_parte01.csv"
]

dfs = {}
for ruta in rutas:
    # Cargamos manejando caracteres problemáticos
    df = pd.read_csv(ruta, sep=';', lineterminator='\n', on_bad_lines='skip')
    
    # Estandarizar nombre de columna (Tiktok tiene 'Content' con mayúscula)
    if 'Content' in df.columns:
        df.rename(columns={'Content': 'content'}, inplace=True)
        
    df = df.dropna(subset=['content'])
    dfs[ruta] = df

df_yt_1 = dfs[rutas[0]].copy()
df_yt_2 = dfs[rutas[1]].copy()
df_tk_1 = dfs[rutas[2]].copy()

print(f"YouTube Parte 1 dimensiones: {df_yt_1.shape}")
print(f"YouTube Parte 2 dimensiones: {df_yt_2.shape}")
print(f"TikTok dimensiones:          {df_tk_1.shape}")

YouTube Parte 1 dimensiones: (6031, 7)
YouTube Parte 2 dimensiones: (4276, 7)
TikTok dimensiones:          (130, 5)


## 4. Funciones de Preprocesamiento

In [4]:
def preprocesamiento_global(texto):
    """
    Limpieza global: 
    - Demojización personalizada al formato 'emoji_nombre'
    - Elimina menciones y URLs.
    """
    if pd.isna(texto):
        return ""
    
    texto = str(texto)
    
    # 1. Demojización personalizada
    texto = emoji.demojize(texto, language='es')
    texto = re.sub(r':([^:]+):', r' emoji_\1 ', texto)
    
    # 2. Eliminar menciones (@usuario)
    texto = re.sub(r'@\w+', ' ', texto)
    
    # 3. Eliminar URLs
    texto = re.sub(r'https?://\S+|www\.\S+', ' ', texto)
    
    # Limpiar espacios múltiples
    texto = re.sub(r'\s+', ' ', texto).strip()
    
    return texto

def pipeline_clasico(texto):
    """
    Pipeline para modelos clásicos:
    - Protege '!' y '?'
    - Minúsculas
    - Limpia puntuación residual
    - Extrae tokens
    - Filtra stopwords
    - Extrae lemas
    """
    # 1. Proteger puntuación clave
    texto = texto.replace('!', ' exclamacion ').replace('¡', ' exclamacion ')
    texto = texto.replace('?', ' interrogacion ').replace('¿', ' interrogacion ')
    
    # 2. Minúsculas
    texto = texto.lower()
    
    # 3. Limpiar caracteres especiales (Mantenemos letras, números y el guion bajo para los emojis)
    texto = re.sub(r'[^a-záéíóúñü0-9_\s]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    
    # 4. Tokenización, Lematización y filtrado de stopwords (usando spaCy con nuestra lista ajustada)
    doc = nlp(texto)
    tokens_originales = []
    lemas_procesados = []
    
    for token in doc:
        palabra = token.text.strip()
        if palabra:
            tokens_originales.append(palabra)
            
        # Ignorar stopwords para los lemas finales
        if token.is_stop:
            continue
            
        lema = token.lemma_.strip()
        if lema:
            lemas_procesados.append(lema)
            
    texto_final = " ".join(lemas_procesados)
    return tokens_originales, lemas_procesados, texto_final

def pipeline_transformers(texto):
    """
    Pipeline para Transformers:
    - Se detiene aquí, dejando el texto casi crudo (con puntuación y mayúsculas)
    """
    # Solo aseguramos que no haya espacios extra residuales de la limpieza global
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

## 5. Aplicación del Pipeline

In [5]:
def procesar_dataframe(df):
    print("  -> Preprocesamiento global...")
    df['content_global'] = df['content'].apply(preprocesamiento_global)
    
    print("  -> Generando versión para modelos clásicos...")
    # La función devuelve una tupla de 3 elementos: (tokens, lemas, texto_final)
    df['tokens_clasico'], df['lemas_clasico'], df['content_clasico'] = zip(*df['content_global'].apply(pipeline_clasico))
    
    print("  -> Generando versión para Transformers...")
    df['content_transformers'] = df['content_global'].apply(pipeline_transformers)
    
    return df

print("Procesando YouTube Parte 1...")
df_yt_1 = procesar_dataframe(df_yt_1)

print("\nProcesando YouTube Parte 2...")
df_yt_2 = procesar_dataframe(df_yt_2)

print("\nProcesando TikTok...")
df_tk_1 = procesar_dataframe(df_tk_1)

Procesando YouTube Parte 1...
  -> Preprocesamiento global...
  -> Generando versión para modelos clásicos...
  -> Generando versión para Transformers...

Procesando YouTube Parte 2...
  -> Preprocesamiento global...
  -> Generando versión para modelos clásicos...
  -> Generando versión para Transformers...

Procesando TikTok...
  -> Preprocesamiento global...
  -> Generando versión para modelos clásicos...
  -> Generando versión para Transformers...


## 6. Verificación de Resultados

In [6]:
pd.set_option('display.max_colwidth', None)

# Mostrar ejemplos de la tokenización y lematización para el modelo clásico
df_yt_1[['content', 'tokens_clasico', 'lemas_clasico', 'content_clasico']].head(3)

,content,tokens_clasico,lemas_clasico,content_clasico
0,"Increíble, Pacto mafioso, ahora quieren apoderarse de Universidades Nacionales; claro, Autoridades MEDIOCRES Q SE PRESTAN","[increíble, pacto, mafioso, ahora, quieren, apoderarse, de, universidades, nacionales, claro, autoridades, mediocres, q, se, prestan]","[increíble, pacto, mafioso, querer, apoderar él, universidad, nacional, autoridad, mediocre, q, prestar]",increíble pacto mafioso querer apoderar él universidad nacional autoridad mediocre q prestar
1,"La universidad del pacífico, la uni...😂","[la, universidad, del, pacífico, la, uni, emoji_cara_llorando_de_risa]","[universidad, pacífico, uni, emoji_cara_llorando_de_risa]",universidad pacífico uni emoji_cara_llorando_de_risa
2,Con eso lo de la presentación de la virgen María ya se quemaron solos 😂😂,"[con, eso, lo, de, la, presentación, de, la, virgen, maría, ya, se, quemaron, solos, emoji_cara_llorando_de_risa, emoji_cara_llorando_de_risa]","[presentación, virgen, maría, quemar, emoji_cara_llorando_de_risa, emoji_cara_llorando_de_risa]",presentación virgen maría quemar emoji_cara_llorando_de_risa emoji_cara_llorando_de_risa


## 7. Exportación

In [7]:
def exportar_datasets(df, ruta_original):
    directorio = os.path.dirname(ruta_original)
    nombre_base = os.path.basename(ruta_original).replace('.csv', '')
    
    # Creamos subcarpeta 'procesado' para mantener orden
    dir_salida = os.path.join(directorio, "procesado")
    os.makedirs(dir_salida, exist_ok=True)
    
    # Exportar archivo Clásico (ahora con las columnas de tokens y lemas explícitas)
    ruta_clasico = os.path.join(dir_salida, f"{nombre_base}_clasico.csv")
    # Retiramos la versión de transformers y temporal
    columnas_clasico = [c for c in df.columns if c not in ['content_global', 'content_transformers']]
    df[columnas_clasico].to_csv(ruta_clasico, sep=';', index=False)
    
    # Exportar archivo Transformers
    ruta_transformers = os.path.join(dir_salida, f"{nombre_base}_transformers.csv")
    columnas_transformers = [c for c in df.columns if c not in ['content_global', 'content_clasico', 'tokens_clasico', 'lemas_clasico']]
    df[columnas_transformers].to_csv(ruta_transformers, sep=';', index=False)
    
    print(f"Exportado: {ruta_clasico}")
    print(f"Exportado: {ruta_transformers}")

print("Iniciando exportación de archivos...")
exportar_datasets(df_yt_1, rutas[0])
exportar_datasets(df_yt_2, rutas[1])
exportar_datasets(df_tk_1, rutas[2])

print("\n¡Proceso de exportación completado!")

Iniciando exportación de archivos...
Exportado: scrapping-youtube-g4\procesado\comentarios_clasificado_parte01_clasico.csv
Exportado: scrapping-youtube-g4\procesado\comentarios_clasificado_parte01_transformers.csv
Exportado: scrapping-youtube-g4\procesado\comentarios_clasificado_parte02_clasico.csv
Exportado: scrapping-youtube-g4\procesado\comentarios_clasificado_parte02_transformers.csv
Exportado: scrapping-tiktok-g4\procesado\comentarios_clasificado_parte01_clasico.csv
Exportado: scrapping-tiktok-g4\procesado\comentarios_clasificado_parte01_transformers.csv

¡Proceso de exportación completado!
